In [2]:
import json
import pandas as pd
from src import data_cleaner
from src.data_validator import generate_quality_report
from src.data_validator import load_rules
from src.data_loader import load_csv
from src.data_validator import check_value_range


In [3]:
path = r"C:\Users\Meta Pc\PycharmProjects\DataSentry\data\dirty_sample.csv"
df = load_csv(path)

In [4]:
df

,product,units_sold,unit_price,region
0,Laptop,5,1200.0,Berlin
1,Mouse,20,25.0,Paris
2,Keyboard,12,75.0,Berlin
3,Monitor,8,300.0,London
4,Laptop,6,1200.0,Paris
5,Mouse,18,NaN,London
6,Keyboard,15,75.0,Berlin
7,Monitor,10,300.0,Paris
8,Laptop,unknown,1200.0,London
9,Mouse,25,25.0,Berlin


In [5]:
with open("../config/data_rules.json", "r") as f:
    j = json.load(f)
j


{'allowed_categories': {'region': ['Berlin', 'Paris', 'London'],
  'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']},
 'value_ranges': {'unit_price': {'min': 0, 'max': 1000},
  'units_sold': {'min': 0, 'max': 1000}},
 'required_columns': ['product', 'units_sold', 'unit_price', 'region'],
 'required_columns_type': {'units_sold': 'numeric',
  'unit_price': 'numeric',
  'product': 'string'}}

In [6]:
rules = load_rules("../config/data_rules.json")
def run_quality_pipeline(path, columns:list, rules):
    df = load_csv(path)
    initial_report = generate_quality_report(df, columns, rules)
    df_copy = df.copy()
    df_copy = data_cleaner.remove_duplicates(df_copy)
    df_copy = data_cleaner.handle_missing_values(df_copy)
    df_copy = data_cleaner.handle_invalid_numeric(df_copy)
    final_report = generate_quality_report(df_copy, columns, rules)
    return (df, initial_report, df_copy, final_report )


In [7]:
df, report, clean, final_report  = run_quality_pipeline(path, ["units_sold","unit_price"], rules)

In [8]:
df

,product,units_sold,unit_price,region
0,Laptop,5,1200.0,Berlin
1,Mouse,20,25.0,Paris
2,Keyboard,12,75.0,Berlin
3,Monitor,8,300.0,London
4,Laptop,6,1200.0,Paris
5,Mouse,18,NaN,London
6,Keyboard,15,75.0,Berlin
7,Monitor,10,300.0,Paris
8,Laptop,unknown,1200.0,London
9,Mouse,25,25.0,Berlin


In [9]:
clean

,product,units_sold,unit_price,region
0,Laptop,5.0,1200.0,Berlin
1,Mouse,20.0,25.0,Paris
2,Keyboard,12.0,75.0,Berlin
3,Monitor,8.0,300.0,London
4,Laptop,6.0,1200.0,Paris
5,Mouse,18.0,25.0,London
6,Keyboard,15.0,75.0,Berlin
7,Monitor,10.0,300.0,Paris
9,Mouse,25.0,25.0,Berlin
10,Keyboard,11.0,75.0,Atlantis


In [10]:
report

{'out_of_range_percentage': {'unit_price': 30.76923076923077,
  'units_sold': 0.0},
 'out_of_range_count': {'unit_price': 4, 'units_sold': 0},
 'missing_columns': [],
 'incorrect_types': [['units_sold', 'numeric']],
 'missing_values_percentage': np.float64(1.9230769230769231),
 'duplicate_percentage': np.float64(7.6923076923076925),
 'missing_values_count': np.int64(1),
 'duplicate_count': np.int64(1),
 'allowed_categories': {},
 'wrong_categories': {},
 'invalid_numeric_count': {'units_sold': 1, 'unit_price': 0},
 'invalid_numeric_percentage': {'units_sold': 7.6923076923076925,
  'unit_price': 0.0},
 'quality_score': np.float64(93.46153846153848)}

In [11]:
final_report

{'out_of_range_percentage': {'unit_price': 18.181818181818183,
  'units_sold': 0.0},
 'out_of_range_count': {'unit_price': 2, 'units_sold': 0},
 'missing_columns': [],
 'incorrect_types': [],
 'missing_values_percentage': np.float64(0.0),
 'duplicate_percentage': np.float64(0.0),
 'missing_values_count': np.int64(0),
 'duplicate_count': np.int64(0),
 'allowed_categories': {},
 'wrong_categories': {},
 'invalid_numeric_count': {'units_sold': 0, 'unit_price': 0},
 'invalid_numeric_percentage': {'units_sold': 0.0, 'unit_price': 0.0},
 'quality_score': np.float64(97.27272727272728)}

In [12]:
j["allowed_categories"]["product"]

['Laptop', 'Mouse', 'Keyboard', 'Monitor']

In [13]:
j["value_ranges"]["units_sold"]["min"]

0

In [14]:
columns = ["region", "product"]
rules = load_rules("../config/data_rules.json")
print(generate_quality_report(df, columns, rules))

{'out_of_range_percentage': {'unit_price': 30.76923076923077, 'units_sold': 0.0}, 'out_of_range_count': {'unit_price': 4, 'units_sold': 0}, 'missing_columns': [], 'incorrect_types': [['units_sold', 'numeric']], 'missing_values_percentage': np.float64(1.9230769230769231), 'duplicate_percentage': np.float64(7.6923076923076925), 'missing_values_count': np.int64(1), 'duplicate_count': np.int64(1), 'allowed_categories': {'region': ['Berlin', 'Paris', 'London'], 'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']}, 'wrong_categories': {'region': region
Atlantis    1
Name: count, dtype: int64, 'product': Series([], Name: count, dtype: int64)}, 'invalid_numeric_count': {'units_sold': 1, 'unit_price': 0}, 'invalid_numeric_percentage': {'units_sold': 7.6923076923076925, 'unit_price': 0.0}, 'quality_score': np.float64(93.46153846153848)}


In [15]:
print(generate_quality_report(df, ["region", "product"], rules))

{'out_of_range_percentage': {'unit_price': 30.76923076923077, 'units_sold': 0.0}, 'out_of_range_count': {'unit_price': 4, 'units_sold': 0}, 'missing_columns': [], 'incorrect_types': [['units_sold', 'numeric']], 'missing_values_percentage': np.float64(1.9230769230769231), 'duplicate_percentage': np.float64(7.6923076923076925), 'missing_values_count': np.int64(1), 'duplicate_count': np.int64(1), 'allowed_categories': {'region': ['Berlin', 'Paris', 'London'], 'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']}, 'wrong_categories': {'region': region
Atlantis    1
Name: count, dtype: int64, 'product': Series([], Name: count, dtype: int64)}, 'invalid_numeric_count': {'units_sold': 1, 'unit_price': 0}, 'invalid_numeric_percentage': {'units_sold': 7.6923076923076925, 'unit_price': 0.0}, 'quality_score': np.float64(93.46153846153848)}


In [16]:
print(check_value_range(df, "unit_price", rules["value_ranges"]["unit_price"]))



4


In [17]:

print(check_value_range(df, "units_sold", rules["value_ranges"]["units_sold"]))

0


In [18]:
from src.quality_pipeline import run_quality_pipeline, generate_cleaning_summary, generate_quality_summary,  generate_final_report
from src.data_validator import load_rules

rules = load_rules("../config/data_rules.json")

df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["region", "product"],
    rules
)

report = generate_cleaning_summary(cleaning_summary)

print(report)

DataSentry Cleaning Report
--------------------------
Original Rows: 13
Duplicates Removed: 1
Rows After Duplicate Removal: 12
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 12
Invalid Numeric Rows Removed: 1
Categories Normalized: 0
Final Rows: 11
Total Rows Removed: 2


In [19]:
quality_report = generate_quality_summary(initial_report)
print(quality_report)

DataSentry Quality Report
-------------------------
Quality Score: 93.46
Missing Values: 1
Duplicate Rows: 1
Invalid Categories: 1
Invalid Numeric Values: 1
Missing Columns: 0
Incorrect Column Types: 1
Out of Range Values: 
unit_price: 4
units_sold: 0

Wrong Categories:
region:
   Atlantis:1
product:
    No invalid categories found




In [20]:
final_report_text = generate_final_report(
    initial_report,
    final_report,
    cleaning_summary
)

print(final_report_text)

DATASENTRY REPORT
[INITIAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 93.46
Missing Values: 1
Duplicate Rows: 1
Invalid Categories: 1
Invalid Numeric Values: 1
Missing Columns: 0
Incorrect Column Types: 1
Out of Range Values: 
unit_price: 4
units_sold: 0

Wrong Categories:
region:
   Atlantis:1
product:
    No invalid categories found



[CLEANING SUMMARY]
DataSentry Cleaning Report
--------------------------
Original Rows: 13
Duplicates Removed: 1
Rows After Duplicate Removal: 12
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 12
Invalid Numeric Rows Removed: 1
Categories Normalized: 0
Final Rows: 11
Total Rows Removed: 2

[FINAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 97.27
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 1
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Types: 0
Out of Range Values: 
unit_price: 2
units_sold: 0

Wrong Categories:
regio

In [21]:
from src import data_cleaner
from src.data_validator import load_rules
rules  = load_rules("../config/data_rules.json")

normalized_df = data_cleaner.normalize_categories(df, rules)

print("Original:", df["product"].unique())
print("Normalized:", normalized_df["product"].unique())

print("Regions:", normalized_df["region"].unique())

Original: <ArrowStringArray>
['Laptop', 'Mouse', 'Keyboard', 'Monitor']
Length: 4, dtype: str
Normalized: <ArrowStringArray>
['Laptop', 'Mouse', 'Keyboard', 'Monitor']
Length: 4, dtype: str
Regions: <ArrowStringArray>
['Berlin', 'Paris', 'London', 'Atlantis']
Length: 4, dtype: str


In [22]:
test_df = df.copy()
test_df.loc[test_df.index[0], "product"] = "keyboard"

normalized_test = data_cleaner.normalize_categories(test_df, rules)

print("Before:", test_df["product"].unique())
print("After:", normalized_test["product"].unique())

Before: <ArrowStringArray>
['keyboard', 'Mouse', 'Keyboard', 'Monitor', 'Laptop']
Length: 5, dtype: str
After: <ArrowStringArray>
['Keyboard', 'Mouse', 'Monitor', 'Laptop']
Length: 4, dtype: str


In [23]:
from src.quality_pipeline import run_quality_pipeline
from src.quality_pipeline import generate_final_report

rules = load_rules("../config/data_rules.json")

df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["product", "region"],
    rules
)

print(generate_final_report(initial_report, final_report, cleaning_summary))

df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["product", "region"],
    rules
)

print("Initial:", initial_report["out_of_range_count"])
print("Final:", final_report["out_of_range_count"])

DATASENTRY REPORT
[INITIAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 93.46
Missing Values: 1
Duplicate Rows: 1
Invalid Categories: 1
Invalid Numeric Values: 1
Missing Columns: 0
Incorrect Column Types: 1
Out of Range Values: 
unit_price: 4
units_sold: 0

Wrong Categories:
product:
    No invalid categories found
region:
   Atlantis:1



[CLEANING SUMMARY]
DataSentry Cleaning Report
--------------------------
Original Rows: 13
Duplicates Removed: 1
Rows After Duplicate Removal: 12
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 12
Invalid Numeric Rows Removed: 1
Categories Normalized: 0
Final Rows: 11
Total Rows Removed: 2

[FINAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 97.27
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 1
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Types: 0
Out of Range Values: 
unit_price: 2
units_sold: 0

Wrong Categories:
produ

In [24]:
from src.data_validator import generate_quality_report
from src.quality_pipeline import load_rules

# Load rules
rules = load_rules("../config/data_rules.json")

# Load dirty dataset
df = pd.read_csv("../data/dirty_sample.csv")

# Generate quality report
report = generate_quality_report(
    df,
    ["product", "region"],
    rules
)

# Check Out of Range Validation
print("Out of Range Count:")
print(report["out_of_range_count"])

print("\nOut of Range Percentage:")
print(report["out_of_range_percentage"])

# Check Quality Score
print("\nQuality Score:")
print(report["quality_score"])

Out of Range Count:
{'unit_price': 4, 'units_sold': 0}

Out of Range Percentage:
{'unit_price': 30.76923076923077, 'units_sold': 0.0}

Quality Score:
93.46153846153848


In [25]:
import pandas as pd
from src.quality_pipeline import generate_quality_report, generate_quality_summary
from src.data_validator import load_rules
rules = load_rules("../config/data_rules.json")

test_df = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard"],
    "units_sold": [5, 20, 12],
    "unit_price": [800, 25, 75],
    "region": ["Berlin", "Paris", "London"]
})

test_report = generate_quality_report(
    test_df,
    ["product", "region"],
    rules
)

print(test_report["out_of_range_count"])
print(generate_quality_summary(test_report))

{'unit_price': 0, 'units_sold': 0}
DataSentry Quality Report
-------------------------
Quality Score: 100.00
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 0
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Types: 0
Out of Range Values: 
No out-of-range values found
Wrong Categories:
product:
    No invalid categories found
region:
    No invalid categories found




In [26]:
import pandas as pd
from src.data_cleaner import normalize_categories
from src.data_validator import load_rules

rules = load_rules("../config/data_rules.json")
test_df = pd.DataFrame({
    "product": [
        "laptop",
        " LAPTOP ",
        "Mouse ",
        "keyboard",
        "Atlantis",
        "Lap top"
    ],
    "region": [
        "berlin",
        " PARIS ",
        "London ",
        "london",
        "Atlantis",
        "Berlin"
    ]
})

normalized_df = normalize_categories(test_df, rules)

print("Original DataFrame:")
print(test_df)

print("\nNormalized DataFrame:")
print(normalized_df)

Original DataFrame:
    product    region
0    laptop    berlin
1   LAPTOP     PARIS 
2    Mouse    London 
3  keyboard    london
4  Atlantis  Atlantis
5   Lap top    Berlin

Normalized DataFrame:
    product    region
0    Laptop    Berlin
1    Laptop     Paris
2     Mouse    London
3  Keyboard    London
4  Atlantis  Atlantis
5   Lap top    Berlin


In [27]:
test_report = generate_quality_report(
    normalized_df,
    ["product", "region"],
    rules
)

print("Wrong Categories:")
print(test_report["wrong_categories"])

Wrong Categories:
{'product': product
Atlantis    1
Lap top     1
Name: count, dtype: int64, 'region': region
Atlantis    1
Name: count, dtype: int64}


In [28]:
import pandas as pd

data = {
    "product": [
        "laptop",
        " LAPTOP ",
        "Atlantis",
        "Lap top",
        "mouse",
        "KEYBOARD"
    ],
    "units_sold": [5, 10, 12, 8, 20, 15],
    "unit_price": [800, 900, 75, 300, 25, 75],
    "region": [
        "berlin",
        " PARIS ",
        "London",
        "Berlin",
        " london",
        "BERLIN"
    ]
}

df = pd.DataFrame(data)

df.to_csv("../data/normalization_test.csv", index=False)



In [29]:
df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/normalization_test.csv",
    ["product", "region"],
    rules
)
print(cleaned_df[["product", "region"]])
print(final_report["wrong_categories"])

    product  region
0    Laptop  Berlin
1    Laptop   Paris
2  Atlantis  London
3   Lap top  Berlin
4     Mouse  London
5  Keyboard  Berlin
{'product': product
Atlantis    1
Lap top     1
Name: count, dtype: int64, 'region': Series([], Name: count, dtype: int64)}


In [31]:
print(cleaning_summary)

{'original_rows': 6, 'after_duplicate_removal': 6, 'duplicates_removed': 0, 'after_handle_missing_value': 6, 'handle_missing_value': 0, 'after_invalid_numeric': 6, 'invalid_numeric': 0, 'after_normalize_categories': 6, 'categories_normalized': np.int64(8)}


In [32]:
print(generate_cleaning_summary(cleaning_summary))

DataSentry Cleaning Report
--------------------------
Original Rows: 6
Duplicates Removed: 0
Rows After Duplicate Removal: 6
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 6
Invalid Numeric Rows Removed: 0
Categories Normalized: 8
Final Rows: 6
Total Rows Removed: 0


In [33]:
print(generate_final_report(initial_report, final_report, cleaning_summary))

DATASENTRY REPORT
[INITIAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 83.33
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 10
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Types: 0
Out of Range Values: 
No out-of-range values found
Wrong Categories:
product:
   laptop:1
    LAPTOP :1
   Atlantis:1
   Lap top:1
   mouse:1
   KEYBOARD:1
region:
   berlin:1
    PARIS :1
    london:1
   BERLIN:1



[CLEANING SUMMARY]
DataSentry Cleaning Report
--------------------------
Original Rows: 6
Duplicates Removed: 0
Rows After Duplicate Removal: 6
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 6
Invalid Numeric Rows Removed: 0
Categories Normalized: 8
Final Rows: 6
Total Rows Removed: 0

[FINAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 96.67
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 2
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Type

In [35]:
df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/sample.csv",
    ["product", "region"],
    rules
)
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)
print("Initial quality score:", initial_report["quality_score"])
print("Final quality score:", final_report["quality_score"])

Original shape: (12, 4)
Cleaned shape: (12, 4)
Initial quality score: 97.5
Final quality score: 97.5


In [36]:
df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["product", "region"],
    rules
)
print("Initial Missing Values:", initial_report["missing_values_count"])
print("Final Missing Values:", final_report["missing_values_count"])

print("Original Shape:", df.shape)
print("Cleaned Shape:", cleaned_df.shape)

print("Cleaning Summary:", cleaning_summary)

Initial Missing Values: 1
Final Missing Values: 0
Original Shape: (13, 4)
Cleaned Shape: (11, 4)
Cleaning Summary: {'original_rows': 13, 'after_duplicate_removal': 12, 'duplicates_removed': 1, 'after_handle_missing_value': 12, 'handle_missing_value': 0, 'after_invalid_numeric': 11, 'invalid_numeric': 1, 'after_normalize_categories': 11, 'categories_normalized': np.int64(0)}


In [37]:
df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/normalization_test.csv",
    ["product", "region"],
    rules
)
print("Initial Wrong Categories:")
print(initial_report["wrong_categories"])

print("\nFinal Wrong Categories:")
print(final_report["wrong_categories"])

print("\nCategories Normalized:")
print(cleaning_summary["categories_normalized"])

Initial Wrong Categories:
{'product': product
laptop      1
 LAPTOP     1
Atlantis    1
Lap top     1
mouse       1
KEYBOARD    1
Name: count, dtype: int64, 'region': region
berlin     1
 PARIS     1
 london    1
BERLIN     1
Name: count, dtype: int64}

Final Wrong Categories:
{'product': product
Atlantis    1
Lap top     1
Name: count, dtype: int64, 'region': Series([], Name: count, dtype: int64)}

Categories Normalized:
8


In [38]:
print("Initial Invalid Numeric:")
print(initial_report["invalid_numeric_count"])

print("\nFinal Invalid Numeric:")
print(final_report["invalid_numeric_count"])

print("\nInvalid Numeric Rows Removed:")
print(cleaning_summary["invalid_numeric"])

Initial Invalid Numeric:
{'units_sold': 0, 'unit_price': 0}

Final Invalid Numeric:
{'units_sold': 0, 'unit_price': 0}

Invalid Numeric Rows Removed:
0


In [39]:
print(df["units_sold"].tolist())

print(pd.to_numeric(
    df["units_sold"],
    errors="coerce"
))

[5, 10, 12, 8, 20, 15]
0     5
1    10
2    12
3     8
4    20
5    15
Name: units_sold, dtype: int64


In [41]:
df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["product", "region"],
    rules
)
print(df["units_sold"].tolist())
print(initial_report["invalid_numeric_count"])
print(final_report["invalid_numeric_count"])
print(cleaning_summary["invalid_numeric"])

['5', '20', '12', '8', '6', '18', '15', '10', 'unknown', '25', '11', '9', '5']
{'units_sold': 1, 'unit_price': 0}
{'units_sold': 0, 'unit_price': 0}
1
